 Automatically classify waste products using image recognition techniques. <br>
 This project aims to build a model that can differentiate between recyclable and organic waste products using transfer learning

In [1]:
import importlib.util
import sys

packages = {
    "numpy": "numpy==2.2.0",
    "pandas": "pandas==2.2.3",
    "sklearn": "scikit-learn==1.6.0",
    "matplotlib": "matplotlib==3.9.3",
    "tensorflow": "tensorflow==2.18.0",
    "seaborn": "seaborn==0.13.2",
    "pyarrow": "pyarrow",
    "requests": "requests",
}

for module_name, pip_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        !{sys.executable} -m pip install {pip_name}
    else:
        print(f"{module_name} is already installed")

numpy is already installed
pandas is already installed
sklearn is already installed
matplotlib is already installed
tensorflow is already installed
seaborn is already installed
pyarrow is already installed
requests is already installed


Importing required libraries

In [2]:
import numpy as np
import os
# import random, shutil
import glob


from matplotlib import pyplot as plt
from matplotlib import pyplot
from matplotlib.image import imread

# from os import makedirs,listdir
# from shutil import copyfile
# from random import seed
# from random import random

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# from tensorflow.keras.layers import Conv2D, MaxPooling2D,GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications import InceptionV3
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')


In [4]:
print(tf.__version__)

2.18.0


I train an algorithm on images and to predict the labels for images in my test set (1 = recyclable, 0 = organic)

In [10]:
import requests
import zipfile
from tqdm import tqdm

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/kd6057VPpABQ2FqCbgu9YQ/o-vs-r-split-reduced-1200.zip"
file_name = "o-vs-r-split-reduced-1200.zip"

print("Downloading file")
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    with open(file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)


def extract_file_with_progress(file_name):
    print("Extracting file with progress")
    with zipfile.ZipFile(file_name, 'r') as zip_ref:
        members = zip_ref.infolist() 
        with tqdm(total=len(members), unit='file') as progress_bar:
            for member in members:
                zip_ref.extract(member)
                progress_bar.update(1)
    print("Finished extracting file")


extract_file_with_progress(file_name)

print("Finished extracting file")
os.remove(file_name)

Extracting file with progress


100%|██████████| 1207/1207 [00:02<00:00, 599.66file/s]

Finished extracting file
Finished extracting file


Model configuration options.

In [11]:
img_rows, img_cols = 150, 150
batch_size = 32
n_epochs = 10
n_classes = 2
val_split = 0.2 #  use 20% of the data for validation purposes
verbosity = 1
path = 'o-vs-r-split/train/'
path_test = 'o-vs-r-split/test/'
input_shape = (img_rows, img_cols, 3)
labels = ['O', 'R']
seed = 42

Loading images using ImageGeneratorClass

folder:
```python
o-vs-r-split/
└── train
    ├── O
    └── R
└── test
    ├── O
    └── R



I create ImageDataGenerators used for training, validation and testing.

Image data generators create batches of tensor image data with real-time data augmentation. The generators loop over the data in batches and are useful in feeding data to the training process.

In [12]:
# Create ImageDataGenerators for training and validation and testing
train_datagen = ImageDataGenerator(
    validation_split = val_split,
    rescale=1.0/255.0,
	width_shift_range=0.1, 
    height_shift_range=0.1, 
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    validation_split = val_split,
    rescale=1.0/255.0,
)

test_datagen = ImageDataGenerator(
    rescale=1.0/255.0
)

In [ ]:
import os

base_dir = os.getcwd()

path = os.path.join(base_dir, "o-vs-r-split", "train")
path_test = os.path.join(base_dir, "o-vs-r-split", "test")

print("Train exists:", os.path.exists(path), path)
print("Test exists:", os.path.exists(path_test), path_test)

In [15]:
train_generator = train_datagen.flow_from_directory(
    directory=path,
    seed=seed,
    batch_size=batch_size,
    class_mode="binary",
    shuffle=True,
    target_size=(img_rows, img_cols),
    subset="training"
)

Found 800 images belonging to 2 classes.


In [16]:
val_generator = val_datagen.flow_from_directory(
    directory = path,
    seed = seed,
    batch_size = batch_size, 
    class_mode='binary',
    shuffle = True,
    target_size=(img_rows, img_cols),
    subset = 'validation'
)

Found 200 images belonging to 2 classes.


Create test generator using test_datagen object

In [17]:
test_generator = test_datagen.flow_from_directory(
    directory=path_test,
    class_mode='binary',
    seed=seed,
    batch_size=batch_size,
    shuffle=False,
    target_size=(img_rows, img_cols)
)

Found 200 images belonging to 2 classes.


length of test_generator

In [18]:
print(len(train_generator))

25
